In [1]:
# ==============================================================================
# 1. 라이브러리 설치 및 데이터 다운로드
# ==============================================================================

!pip install efficientnet_pytorch
!pip install torchmetrics
!pip install kagglehub  # KaggleHub 설치

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os
import kagglehub # 임포트
import time
import copy
from google.colab import drive
from collections import Counter # ⚠️ 클래스 카운트를 위한 라이브러리 추가
from google.colab import drive #

# 다운로드 및 경로 설정
path = kagglehub.dataset_download("nizier193/strawberry-disease-classification")

print("다운로드된 데이터셋 경로:", path)

# GPU 사용 설정
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

  Preparing metadata (setup.py) ... done
  Created wheel for efficientnet_pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16426 sha256=ab847b227ee45d552b5a939cc5c3ad367e3989c9bae283492de5360c01ac30de
  Stored in directory: /root/.cache/pip/wheels/9c/3f/43/e6271c7026fe08c185da2be23c98c8e87477d3db63f41f32ad
Successfully built efficientnet_pytorch
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 21.7 MB/s eta 0:00:00
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


100%|██████████| 7.82G/7.82G [01:29<00:00, 94.1MB/s]

Extracting files...


다운로드된 데이터셋 경로: /root/.cache/kagglehub/datasets/nizier193/strawberry-disease-classification/versions/1
사용 장치: cuda:0


In [2]:
# ==============================================================================
# 2. 데이터셋 경로 설정 (이미지 확인 후 수정됨)
# ==============================================================================
import os

# 'path'는 1번 섹션에서 다운로드한 루트 경로입니다.
DATA_ROOT_DOWNLOAD = path
print(f"DEBUG: 다운로드 루트 경로: {DATA_ROOT_DOWNLOAD}")

# 1. 훈련용 데이터가 있는 폴더 이름 (이미지에서 확인한 폴더명)
TARGET_FOLDER = 'union_dataset'

# 2. 경로 결합
DATA_ROOT_FOR_SPLIT = os.path.join(DATA_ROOT_DOWNLOAD, TARGET_FOLDER)

print(f"\n✅ 최종 데이터 로드 경로 (DATA_ROOT_FOR_SPLIT): {DATA_ROOT_FOR_SPLIT}")

# 3. 최종 경로 검증
if not os.path.exists(DATA_ROOT_FOR_SPLIT):
    print(f"❌ 오류: 경로를 찾을 수 없습니다. : {DATA_ROOT_FOR_SPLIT}")
else:
    print(f"✅ 경로 확인 완료: {DATA_ROOT_FOR_SPLIT}")
    print("     '4. 데이터 로드' 섹션을 실행하세요.")

DEBUG: 다운로드 루트 경로: /root/.cache/kagglehub/datasets/nizier193/strawberry-disease-classification/versions/1

✅ 최종 데이터 로드 경로 (DATA_ROOT_FOR_SPLIT): /root/.cache/kagglehub/datasets/nizier193/strawberry-disease-classification/versions/1/union_dataset
✅ 경로 확인 완료: /root/.cache/kagglehub/datasets/nizier193/strawberry-disease-classification/versions/1/union_dataset
     '4. 데이터 로드' 섹션을 실행하세요.


In [3]:
# ==============================================================================
# 3. 하이퍼파라미터 및 데이터 변환 설정 (모델 축소 적용)
# ==============================================================================
from efficientnet_pytorch import EfficientNet

# 하이퍼파라미터
BATCH_SIZE = 32 # B0 모델에 맞춰 배치 사이즈를 32로 늘려 속도 개선
NUM_EPOCHS = 10
LEARNING_RATE = 0.0001 # 과적합 방지를 위해 학습률은 계속 낮게 유지
MODEL_NAME = 'efficientnet-b0' # ⭐️ 모델을 EfficientNet-B0로 축소

# EfficientNet-B0의 권장 입력 이미지 크기 (224x224로 자동 변경됨)
INPUT_SIZE = EfficientNet.get_image_size(MODEL_NAME)

# 데이터 증강 및 정규화
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(INPUT_SIZE),
        transforms.RandomResizedCrop(INPUT_SIZE),
        # 💡 [추가] 회전 및 색상 변화 추가
        transforms.RandomRotation(degrees=20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # ImageNet 통계
    ]),
    'test': transforms.Compose([
        transforms.Resize(INPUT_SIZE),
        transforms.CenterCrop(INPUT_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}


In [4]:
# ==============================================================================
# 4. 데이터 로드 및 DataLoader 생성 (수정: 단일 폴더 구조 → Random Split)
# ==============================================================================
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split

print("\n데이터 로딩 및 안전한 90:10 분할 중...")

# 1. DATA_ROOT 전체를 하나의 데이터셋으로 로드합니다.
# ⚠️ [수정] DATA_ROOT 대신, 클래스 폴더가 들어있는 정확한 경로 변수를 사용해야 합니다.
#    이전 경로 설정 섹션에서 정의한 DATA_ROOT_FOR_SPLIT 변수를 사용합니다.
full_dataset = ImageFolder(DATA_ROOT_FOR_SPLIT, data_transforms['train']) # 💡 DATA_ROOT_FOR_SPLIT 사용
full_size = len(full_dataset)

# 2. 비율 설정 (90% 학습, 10% 검증)
train_size = int(0.9 * full_size)
val_size = full_size - train_size

# 3. 데이터셋 분리 (겹치지 않음을 보장)
train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size]
)

# 4. DataLoader 재정의
dataloaders = {
    'train': DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True),
    # 'val' 키에 검증 데이터셋(val_dataset)을 로드합니다.
    'val': DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
}

# 5. 메타 정보 업데이트
dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset)}
class_names = full_dataset.classes
NUM_CLASSES = len(class_names)

print(f"클래스 개수: {NUM_CLASSES}")
print(f"클래스 이름: {class_names}")
print(f"✅ 새로운 학습 데이터 크기: {dataset_sizes['train']} (90%)")
print(f"✅ 새로운 검증 데이터 크기: {dataset_sizes['val']} (10%)")


데이터 로딩 및 안전한 90:10 분할 중...
클래스 개수: 77
클래스 이름: ['Strawberry___Leaf_scorch', 'Strawberry___healthy', 'alternaria_leaf_blight', 'angular_leafspot', 'anthocyanosis', 'anthracnose', 'anthracnose_fruit_rot', 'ants', 'aphid', 'aphid_effects', 'ascochyta_blight', 'bacterial_spot', 'black_chaff', 'black_rot', 'black_spots', 'blossom_blight', 'blossom_end_rot', 'botrytis_cinerea', 'burn', 'calciumdeficiency', 'canker', 'caterpillars', 'cherry_leaf_spot', 'coccomyces_of_pome_fruits', 'colorado_beetle', 'colorado_beetle_effects', 'corn_downy_mildew', 'cyclamen_mite', 'downy_mildew', 'dry_rot', 'edema', 'esca', 'eyespot', 'frost_cracks', 'galls', 'gray_mold', 'grey_mold', 'gryllotalpa', 'gryllotalpa_effects', 'healthy', 'late_blight', 'leaf_deformation', 'leaf_miners', 'leaf_spot', 'leaves_scorch', 'lichen', 'loss_of_foliage_turgor', 'marginal_leaf_necrosis', 'mealybug', 'mechanical_damage', 'monilia', 'mosaic_virus', 'northern_leaf_blight', 'nutrient_deficiency', 'pear_blister_mite', 'pest_damage

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [5]:
# ==============================================================================
# 5. 모델 설정 및 가중치 부여 손실 함수 설정 (✨ B0 모델로 축소)
# ==============================================================================
from collections import Counter
import torch.nn as nn
from efficientnet_pytorch import EfficientNet

# ==============================================================================
# 1. 클래스 가중치 계산 (Class Imbalance 대처)
# ==============================================================================
from collections import Counter
# ⚠️ image_datasets['train'] 대신 full_dataset을 사용합니다.
# full_dataset은 '4. 데이터 로드' 섹션에서 정의되어야 합니다.

# full_dataset은 Subset이 아닌 ImageFolder 객체여야 target을 가져올 수 있습니다.
# 만약 full_dataset이 ImageFolder 객체였다면:
# train_targets = full_dataset.targets

# 🚨 full_dataset이 random_split 전의 ImageFolder 객체였고, 그 객체가 image_datasets['train']에 해당한다면
# 쌀 모델에서 'train' 데이터셋만 사용했으므로,
# '4. 데이터 로드' 섹션에서 정의된 `full_dataset` 객체를 사용해야 합니다.
# 현재 구조에서는 full_dataset이 ImageFolder 객체이므로 targets 속성이 있습니다.

train_targets = full_dataset.targets # 💡 [수정] full_dataset 객체를 사용해야 합니다.
class_counts = Counter(train_targets)
total_count = sum(class_counts.values())

class_weights = [total_count / class_counts[i] for i in range(NUM_CLASSES)]
class_weights = torch.tensor(class_weights, dtype=torch.float)
class_weights = class_weights / class_weights.min()

print(f"✅ 클래스 가중치 계산 완료: {class_weights.tolist()}")


# ----------------------------------------------------------------------
# 2. EfficientNet-B0 모델 로드 및 분류 레이어 명시적 교체 (수정)
# ----------------------------------------------------------------------

# ImageNet 가중치로 EfficientNet-B0 모델을 로드합니다.
# MODEL_NAME 변수는 이제 'efficientnet-b0'입니다.
model_ft = EfficientNet.from_pretrained(MODEL_NAME).to(device)

# 기존 분류 레이어(_fc)의 입력 피처 수 확인
num_ftrs = model_ft._fc.in_features

# 💡 [수정] 실제 데이터셋의 클래스 수에 맞춰 새로운 Sequential 레이어 정의 및 교체
#      (Dropout(0.5)을 추가하여 과적합 대처)
model_ft._fc = nn.Sequential(
    nn.Dropout(0.5), # 50%의 확률로 Dropout 적용
    nn.Linear(num_ftrs, NUM_CLASSES)
)

model_ft = model_ft.to(device) # 장치에 다시 올립니다.

print(f"✅ EfficientNet-B0의 최종 분류 레이어(w/ Dropout)가 {num_ftrs} -> {NUM_CLASSES}로 성공적으로 교체되었습니다.")


# ----------------------------------------------------------------------
# 3. Loss, Optimizer, Scheduler 정의
# ----------------------------------------------------------------------

# 계산된 class_weights를 GPU/CPU 장치에 올리고 CrossEntropyLoss에 적용
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# AdamW 옵티마이저 사용 (수정된 LEARNING_RATE 적용)
optimizer_ft = optim.AdamW(model_ft.parameters(), lr=LEARNING_RATE)

# 학습률 스케줄러: 8 에포크마다 0.5배 감소로 완화
exp_lr_scheduler = optim.lr_scheduler.StepLR(optimizer_ft, step_size=8, gamma=0.5)

print("✅ Loss, Optimizer, Scheduler 정의 완료.")


✅ 클래스 가중치 계산 완료: [1.0, 2.4320175647735596, 28.435895919799805, 2.8290815353393555, 36.96666717529297, 32.61764907836914, 13.04705810546875, 29.972970962524414, 24.108694076538086, 16.803030014038086, 25.204544067382812, 20.16363525390625, 29.972970962524414, 29.972970962524414, 27.04878044128418, 6.195530414581299, 24.644445419311523, 58.3684196472168, 8.801587104797363, 1.3776397705078125, 30.805553436279297, 25.79069709777832, 19.120689392089844, 30.805553436279297, 31.685714721679688, 39.60714340209961, 30.805553436279297, 38.24137878417969, 25.204544067382812, 15.619717597961426, 18.79660987854004, 15.191780090332031, 29.972970962524414, 23.595745086669922, 16.55223846435547, 2.7725000381469727, 14.402596473693848, 44.36000061035156, 42.653846740722656, 1.357405185699463, 15.402776718139648, 16.803030014038086, 14.5921049118042, 1.8607381582260132, 25.79069709777832, 17.603174209594727, 14.03797435760498, 7.809858798980713, 14.786665916442871, 19.456140518188477, 34.65625, 14.59210

100%|██████████| 20.4M/20.4M [00:00<00:00, 270MB/s]


Loaded pretrained weights for efficientnet-b0
✅ EfficientNet-B0의 최종 분류 레이어(w/ Dropout)가 1280 -> 77로 성공적으로 교체되었습니다.
✅ Loss, Optimizer, Scheduler 정의 완료.


In [6]:
# ==============================================================================
# 6. 모델 학습 함수 (수정됨: 'test'를 'val'로 변경)
# ==============================================================================
def train_model(model, criterion, optimizer, scheduler, num_epochs=10):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        epoch_start_time = time.time()
        print(f'\nEpoch {epoch+1}/{num_epochs}'); print('-' * 10)

        # 💡 [수정] 'test' 대신 'val' 단계를 반복합니다.
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            # 💡 [수정] 출력 메시지에서 'test'를 'val'로 변경
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # 💡 [수정] 'val' 단계의 정확도가 최고 기록일 때 모델을 저장합니다.
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                print(f'-> New best model found! Saving weights with Acc: {best_acc:.4f}')

        epoch_time = time.time() - epoch_start_time
        print(f'Epoch Time: {epoch_time:.0f}초')

    time_elapsed = time.time() - since
    print(f'\n학습 완료! 총 시간: {time_elapsed // 60:.0f}분 {time_elapsed % 60:.0f}초')
    print(f'최고 검증 정확도: {best_acc:.4f}')

    model.load_state_dict(best_model_wts)
    return model

In [7]:
# ==============================================================================
# 7. 모델 학습 실행 및 저장
# ==============================================================================

# 모델 학습 시작
model_ft = train_model(model_ft, criterion, optimizer_ft, exp_lr_scheduler, num_epochs=NUM_EPOCHS)

# Google Drive 마운트
drive.mount('/content/drive')
MODEL_SAVE_PATH = '/content/drive/MyDrive/disease_model.pth' # ⚠️ 파일 이름에 weighted 추가

# 모델 저장
torch.save(model_ft.state_dict(), MODEL_SAVE_PATH)
print(f"\n✅ 최적 모델 가중치가 {MODEL_SAVE_PATH}에 저장되었습니다.")


Epoch 1/10
----------
train Loss: 4.1721 Acc: 0.2063
val Loss: 3.7793 Acc: 0.4810
-> New best model found! Saving weights with Acc: 0.4810
Epoch Time: 403초

Epoch 2/10
----------
train Loss: 3.5675 Acc: 0.4806
val Loss: 3.0505 Acc: 0.5789
-> New best model found! Saving weights with Acc: 0.5789
Epoch Time: 404초

Epoch 3/10
----------
train Loss: 2.7418 Acc: 0.5786
val Loss: 2.3128 Acc: 0.6474
-> New best model found! Saving weights with Acc: 0.6474
Epoch Time: 403초

Epoch 4/10
----------
train Loss: 2.1402 Acc: 0.6410
val Loss: 1.9019 Acc: 0.6736
-> New best model found! Saving weights with Acc: 0.6736
Epoch Time: 403초

Epoch 5/10
----------
train Loss: 1.8053 Acc: 0.6832
val Loss: 1.7686 Acc: 0.7018
-> New best model found! Saving weights with Acc: 0.7018
Epoch Time: 404초

Epoch 6/10
----------
train Loss: 1.5763 Acc: 0.7054
val Loss: 1.6077 Acc: 0.7138
-> New best model found! Saving weights with Acc: 0.7138
Epoch Time: 406초

Epoch 7/10
----------
train Loss: 1.4115 Acc: 0.7295
val 

In [10]:
# ==============================================================================
# 8. ONNX 모델 변환 및 저장 (별도 블럭)
# ==============================================================================

!pip install onnxscript
!pip install onnx
!pip install onnxruntime

# ⚠️ '7. 모델 학습 실행' 블럭이 실행된 후,
# ⚠️ model_ft, INPUT_SIZE, device 변수가 메모리에 로드된 상태여야 합니다.

ONNX_SAVE_PATH = 'strawberry_disease_model.onnx'

# 1. 모델을 '평가 모드'로 설정 (필수!)
# Dropout, BatchNorm 등이 추론 모드로 변경됩니다.
model_ft.eval()

# 2. 모델에 입력될 더미 텐서(dummy input) 생성
# (EfficientNet-B0 기준 INPUT_SIZE = 224)
# (Batch_size, Channels, Height, Width)
dummy_input = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE, device=device)

# 3. ONNX로 내보내기
try:
    torch.onnx.export(
        model_ft,                   # 실행될 모델 (메모리에 로드된 상태)
        dummy_input,                # 모델 입력값 (모양과 타입)
        ONNX_SAVE_PATH,             # 저장될 ONNX 파일 경로
        export_params=True,         # 모델 파라미터(가중치) 저장
        opset_version=11,           # ONNX 오퍼레이터 버전
        do_constant_folding=True,   # 최적화
        input_names = ['input'],    # ONNX 그래프의 입력 텐서 이름
        output_names = ['output'],  # ONNX 그래프의 출력 텐서 이름
        dynamic_axes={'input' : {0 : 'batch_size'},    # 배치 크기를 동적으로 설정
                      'output' : {0 : 'batch_size'}}
    )
    print(f"\n✅ 모델이 ONNX 형식으로 {ONNX_SAVE_PATH}에 성공적으로 저장되었습니다.")

except Exception as e:
    print(f"\n❌ ONNX 변환 중 오류 발생: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 683.0/683.0 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 14.0 MB/s eta 0:00:00


/tmp/ipython-input-766267436.py:25: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1126 09:07:51.784000 1092 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 127, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 122, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/BaseConverter.h:65: adapter_lookup: Assertion `false`

Applied 115 of general pattern rewrite rules.

✅ 모델이 ONNX 형식으로 strawberry_disease_model.onnx에 성공적으로 저장되었습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')